In [1]:
from dataclasses import dataclass
from typing import Dict

import pandas as pd

from dotenv import load_dotenv
from openai import OpenAI
from typesafe_sdk import TypeSafeClient, Noul

# Carregar chaves de API do .env
load_dotenv()

True

# Loop de Colaboração: Jev + OpenAI

Refinamento iterativo de soluções: OpenAI propõe, Jev avalia, feedback impulsiona próxima iteração.
Demonstra **resolução de problemas multi-agente** onde modelos especializados trabalham em conjunto.

## Fluxo
1. OpenAI gera solução inicial ou refina baseado em feedback
2. Jev avalia contra 5 critérios (correto, completo, viável, claro, simples)
3. Se todos critérios ≥ 0.80 confiança: convergência atingida, parar
4. Senão: construir feedback dos critérios falhados, iterar

In [2]:
problem = """
Uma empresa possui um sistema de atendimento ao cliente.

Atualmente, todos os tickets entram em uma única fila.
A empresa quer automatizar a triagem.

Proponha uma arquitetura que:

1. classifique os tickets por departamento;
2. identifique tickets urgentes;
3. identifique risco de cancelamento;
4. mantenha um humano no loop para casos incertos;
5. seja simples o suficiente para uma equipe pequena manter.
"""

## Enunciado do Problema

Sistema de atendimento ao cliente com fila única. Objetivo: automatizar roteamento sem perder supervisão humana para casos incertos.

## Rubrica de Avaliação

Cinco critérios guiam avaliação de Jev. Cada um marca 0.0–1.0 de confiança que critério foi satisfeito.

In [3]:
JUDGE_CRITERIA = {
    "correctness": (
        "The proposed solution is logically correct and does not "
        "contain important technical contradictions."
    ),

    "completeness": (
        "The solution addresses all requirements stated in the problem."
    ),

    "feasibility": (
        "The proposed solution is realistically implementable by "
        "a small engineering team."
    ),

    "clarity": (
        "The solution is clear, structured, and sufficiently specific "
        "to be actionable."
    ),

    "simplicity": (
        "The architecture avoids unnecessary complexity and is appropriate "
        "for the stated constraints."
    ),
}

In [4]:
openai_client = OpenAI()

OPENAI_MODEL = "gpt-5-mini"

## Configuração de Modelos

## OpenAI: Propor ou Refinar

Primeira chamada: início frio apenas do problema. Subsequentes: refinamento iterativo baseado em feedback de Jev.

In [5]:
def ask_openai(
    problem: str,
    previous_solution: str | None = None,
    feedback: str | None = None,
) -> str:
    # Início frio: gerar solução apenas do problema
    if previous_solution is None:
        prompt = f"""
Você é responsável por resolver o problema abaixo.

PROBLEMA:
{problem}

Produza uma solução clara, tecnicamente consistente e prática.

Não discuta o processo de avaliação.
Entregue somente a solução proposta.
"""
    # Refinamento iterativo: ajustar baseado em feedback de Jev
    else:
        prompt = f"""
Você está revisando uma solução anterior.

PROBLEMA:
{problem}

SOLUÇÃO ANTERIOR:
{previous_solution}

FEEDBACK DO AVALIADOR:
{feedback}

Produza uma nova versão da solução.

Regras:
- Corrija os problemas apontados pelo feedback.
- Preserve as partes da solução que já estavam boas.
- Não mencione o avaliador.
- Não explique o processo de revisão.
- Entregue apenas a solução final revisada.
"""

    response = openai_client.responses.create(
        model=OPENAI_MODEL,
        input=prompt,
    )

    return response.output_text

## Jev: Avaliação Estruturada

Avalia solução proposta contra cada critério independentemente via julgamentos sim/não binários.
Confiança de Jev (Noul) convertida para 0.0–1.0 de pontuação por critério.

In [6]:
def evaluate_with_jev(
    problem: str,
    solution: str,
) -> Dict[str, float]:
    # Estado fornecido para Jev ter contexto
    state = {
        "problem": problem,
        "proposed_solution": solution,
    }

    # Construir questões: um Noul por critério
    questions = {
        name: Noul(
            instructions=(
                f"Evaluate `proposed_solution` for the following criterion: "
                f"{description} "
                "Consider the requirements in `problem`."
            ),
            criteria={
                "true": (
                    "The proposed solution satisfies this criterion "
                    "to a strong degree."
                ),
                "false": (
                    "The proposed solution fails to satisfy this criterion "
                    "or has important deficiencies related to it."
                ),
            },
        )
        for name, description in JUDGE_CRITERIA.items()
    }

    # Consultar Jev em paralelo em todos critérios
    with TypeSafeClient() as client:
        response = client.system_one(
            state=state,
            questions=questions,
        )

    # Extrair probabilidades por critério
    return {
        name: response.answers[name].noul
        for name in questions
    }

## Construir Feedback a Partir de Scores

Identificar critérios falhados (score < threshold). Construir feedback acionável para próxima iteração.
Se todos passam, sinalizar convergência.

In [7]:
def build_feedback(
    scores: Dict[str, float],
    threshold: float = 0.80,
) -> str:
    # Coletar critérios falhados
    problems = []

    for criterion, score in scores.items():
        # Critério abaixo do threshold: adicionar ao feedback
        if score < threshold:
            description = JUDGE_CRITERIA[
                criterion
            ]

            problems.append(
                f"- Improve {criterion}: "
                f"{description} "
                f"Current confidence that the criterion is satisfied: "
                f"{score:.2f}."
            )

    # Todos passam: convergência atingida
    if not problems:
        return (
            "All evaluation criteria are currently satisfied "
            "above the required threshold."
        )

    # Critérios falhando: estruturar feedback para próxima iteração
    return (
        "The solution needs improvement in the following areas:\n"
        + "\n".join(problems)
    )

## Verificação de Convergência

Critério: todos scores ≥ threshold. Parar loop quando atingido.

In [8]:
def passes_evaluation(
    scores: Dict[str, float],
    threshold: float = 0.80,
) -> bool:
    # Convergido: todos critérios atingem ou superam threshold
    return all(
        score >= threshold
        for score in scores.values()
    )

## Estado da Colaboração

Rastrear cada rodada: proposta, scores, feedback, status pass/fail.

In [9]:
@dataclass
class CollaborationRound:
    round_number: int
    solution: str  # Proposta de OpenAI
    scores: Dict[str, float]  # Avaliações de Jev por critério
    feedback: str  # Feedback acionável para próxima iteração
    passed: bool  # Convergido?

## Loop Principal: Refinamento Iterativo

Executar rodadas: OpenAI propõe → Jev avalia → construir feedback → iterar.

Cada rodada:
1. **Propor**: OpenAI gera ou refina solução
2. **Avaliar**: Jev marca em 5 critérios
3. **Decidir**: Se convergido (todos scores ≥ threshold), parar. Senão, iterar.

Para em convergência ou `max_rounds`, o que vier primeiro.

In [10]:
def collaborate(
    problem: str,
    max_rounds: int = 5,
    threshold: float = 0.80,
):
    history = []
    solution = None
    feedback = None

    for round_number in range(
        1,
        max_rounds + 1,
    ):
        print(
            f"\n{'=' * 70}"
        )
        print(
            f"RODADA {round_number}"
        )
        print(
            f"{'=' * 70}"
        )

        # -----------------------------------
        # Passo 1: OpenAI propõe ou refina
        # -----------------------------------
        solution = ask_openai(
            problem=problem,
            previous_solution=solution,
            feedback=feedback,
        )

        print("\\nOPENAI:")
        print(solution)

        # -----------------------------------
        # Passo 2: Jev avalia solução
        # -----------------------------------
        scores = evaluate_with_jev(
            problem=problem,
            solution=solution,
        )

        print("\\nJEV:")

        for criterion, score in scores.items():
            print(
                f"{criterion:15s}: "
                f"{score:.3f}"
            )

        # -----------------------------------
        # Passo 3: Verificar convergência e construir feedback
        # -----------------------------------
        passed = passes_evaluation(
            scores,
            threshold=threshold,
        )

        feedback = build_feedback(
            scores,
            threshold=threshold,
        )

        print("\\nFEEDBACK:")
        print(feedback)

        # Armazenar estado da rodada
        history.append(
            CollaborationRound(
                round_number=round_number,
                solution=solution,
                scores=scores,
                feedback=feedback,
                passed=passed,
            )
        )

        # Parar se convergido
        if passed:

            print(
                "\\nCritérios atingidos. "
                "Colaboração encerrada."
            )

            break

    return solution, history

## Executar Colaboração

Rodar loop de colaboração: máx 5 rodadas, threshold 0.80.

In [11]:
final_solution, history = collaborate(
    problem=problem,
    max_rounds=5,
    threshold=0.80,
)


RODADA 1
\nOPENAI:
Solução proposta (arquitetura prática, simples e mantível por equipe pequena)

Visão geral
- Sistema híbrido: regras + modelos ML leves (multi-task) para classificar departamento, detectar urgência e risco de cancelamento, com humano no loop para casos incertos.
- Objetivos operacionais: alta captura (recall) de urgentes e risco de cancelamento; roteamento automático quando confiança for alta; revisão humana para incertezas e aprendizagem contínua.
- Preferir ferramentas e deploys simples e gerenciáveis: containerização (Docker), API leve (FastAPI), banco relacional (Postgres), fila simples (Redis Streams / RabbitMQ) e modelo transformador pequeno (DistilBERT / BERTimbau) ou um ensemble de regras+modelo.

Arquitetura (componentes e fluxo)
1. Ingestão
   - Endpoint HTTP/ webhook que recebe tickets (texto, metadados: canal, cliente, tempo, contratos).
   - Normalização e enriquecimento (extrair idioma, tokens, entidades básicas — nome, produto).
2. Pré-processamento
 

## Resultados da Colaboração

Resumir convergência: número de rodada, todos 5 scores de critério, flag pass/fail.

In [12]:
rows = []

# Achatar histórico em linhas de dataframe
for item in history:
    row = {
        "round": item.round_number,
        "passed": item.passed,
    }

    # Adicionar todos scores de critério como colunas
    row.update(
        item.scores
    )

    rows.append(row)

# Exibir tabela de convergência
history_df = pd.DataFrame(rows)

history_df

,round,passed,correctness,completeness,feasibility,clarity,simplicity
0,1,False,0.93,0.97,0.80,0.94,0.67
1,2,True,0.90,0.97,0.86,0.95,0.85
